```bash
pip install pandas nltk vaderSentiment textblob
python -m textblob.download_corpora
```

In [1]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

In [2]:
df = pd.read_csv(r"../raw/1000-US-Ind-Aus.csv")
df.head()

,headline,country,label
0,0% salary hike even after great performance? E...,India,Sensational/Clickbait
1,Learjet 45: The jet Ajit Pawar took for his fi...,India,Human-Interest
2,Lead the next wave of consumer businesses with...,India,Promotional
3,CUET UG 2026 registration ends in two days: Ho...,India,Neutral/Factual
4,Employee who was denied WFH by boss due to los...,India,Human-interest


In [3]:
analyzer = SentimentIntensityAnalyzer()

BIAS_WORDS = [
    "disastrous","shocking","failure","chaos","outrage","reckless",
    "disgrace","collapse","crisis","scandal","slam","blasts",
    "furious","radical","damaging","toxic","controversial"
]

NEUTRAL_WORDS = [
    "announces","reports","states","confirms","according",
    "official","statement","data","meeting","release","survey"
]

In [4]:
def score_headline(text):
    text_lower = text.lower()

    vader = analyzer.polarity_scores(text)["compound"]
    subjectivity = TextBlob(text).sentiment.subjectivity

    bias_hits = sum(1 for w in BIAS_WORDS if re.search(rf"\b{w}\b", text_lower))
    neutral_hits = sum(1 for w in NEUTRAL_WORDS if re.search(rf"\b{w}\b", text_lower))

    bias_score = (
        abs(vader) * 0.4 +
        subjectivity * 0.4 +
        bias_hits * 0.2
    )

    neutral_score = neutral_hits * 0.5

    return bias_score, neutral_score

In [5]:
def assign_label(text):
    bias_score, neutral_score = score_headline(text)

    if bias_score >= 0.45:
        return "biased"
    elif neutral_score >= 0.5:
        return "neutral"
    else:
        return "neutral"  # default safe label

In [6]:
df["label"] = df["headline"].apply(assign_label)

print(df["label"].value_counts())
df.to_csv("headlines_silver_labeled.csv", index=False)

print("✅ Fully automated silver-label dataset created.")

label
neutral    2644
biased      325
Name: count, dtype: int64
✅ Fully automated silver-label dataset created.
